# 03 — Modelagem e Experimentos

**MedAI Triage** · Fase 4 do desafio

Seis modelos, dois feature sets, doze experimentos. Este notebook decide **qual modelo** e **em que ponto de operação** o sistema vai funcionar.

A decisão mais difícil da fase não é qual algoritmo usar — é **como traduzir a prioridade clínica em critério automático de seleção**. Recall é a prioridade; mas otimizar recall diretamente seleciona o modelo que classifica tudo como maligno. A seção 2 trata disso em detalhe.

| Seção | Pergunta |
|---|---|
| 1 | Quais modelos, e por quê cada um? |
| 2 | Como selecionar sem cair na solução degenerada? |
| 3 | Qual venceu? |
| 4 | O feature set reduzido custou desempenho? |
| 5 | As probabilidades são confiáveis? |
| 6 | Onde cortar? |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from medai import viz
from medai.config import carregar_config
from medai.models import calibrate, registry
from medai.models.evaluate import (
    classificar_com_faixa, escolher_limiar, metricas_classificacao, resumo_faixas,
)
from medai.models.train import (
    CRITERIO_SELECAO, construir_feature_sets, executar_experimentos,
    preparar_dados, selecionar_campeao,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
viz.configurar_estilo(interativo=True)

config = carregar_config(RAIZ / "configs" / "breast_cancer.yaml")
dados = preparar_dados(config)
print(dados)

---
## 1. O catálogo de modelos

Cada entrada do catálogo declara **por que está ali**. O desafio pede duas ou mais técnicas; usamos seis para permitir comparação real, incluindo um baseline que não aprende nada.

In [ ]:
for especificacao in registry.listar(config.modelagem.modelos):
    print(f"■ {especificacao.nome}  ({especificacao.busca}, {especificacao.n_combinacoes} combinações)")
    print(f"  {especificacao.justificativa}\n")

---
## 2. O critério de seleção — e um erro que foi corrigido

### O problema

Recall da classe maligna é a métrica primária por razão clínica legítima (ADR-003). Mas usá-la como critério de seleção automática tem uma consequência indesejada: **o classificador que responde "maligno" para todos os casos obtém recall 1,0**. Ele nunca deixa passar um câncer — porque nunca deixa passar nada.

In [ ]:
from sklearn.metrics import f1_score, fbeta_score, precision_score, recall_score

y = np.array([0] * 63 + [1] * 37)       # prevalência de 37%, como no dataset
degenerado = np.ones(100, dtype=int)     # "maligno" para tudo
bom = np.array([0] * 60 + [1] * 3 + [1] * 36 + [0] * 1)

comparacao = pd.DataFrame([
    {
        "classificador": nome,
        "recall": recall_score(y, p),
        "precision": precision_score(y, p),
        "F1": f1_score(y, p),
        "F2": fbeta_score(y, p, beta=2),
    }
    for nome, p in [("degenerado (tudo maligno)", degenerado), ("bom modelo", bom)]
]).set_index("classificador")
comparacao.round(3)

### A tentativa que falhou

A primeira implementação usava a **regra de um erro padrão**: aceitar os candidatos cujo recall médio estivesse dentro de um desvio do melhor e, entre eles, escolher o de maior F1.

Parecia razoável. **Falha exatamente no caso que deveria cobrir:** o classificador degenerado acerta todos os positivos em *todos* os folds, então seu desvio padrão é exatamente zero. A janela de equivalência colapsa para um único ponto e ele vence sozinho.

O defeito foi descoberto por um teste escrito para verificar a proteção — o teste falhou, e o problema era do código, não do teste.

Ajustar a tolerância para um valor fixo não resolveria de forma honesta: o número escolhido passaria a carregar toda a decisão, sem nada que o sustentasse.

### A solução

**F-beta com β = 2**, que pondera recall β² = 4 vezes mais que precisão.

Como a tabela acima mostra, o degenerado obtém F2 = 0,746 contra 0,963 do bom modelo. A assimetria de custo fica embutida na definição da métrica, em vez de ser reconstruída com regras de desempate.

**E a garantia de recall ≥ 98%?** Vem do ajuste de limiar (seção 6), que é onde ela pertence. Escolher a *família de modelo* e escolher o *ponto de operação* são decisões distintas — confundi-las foi a origem do problema. Registrado em ADR-014.

---
## 3. Execução dos experimentos

Seis modelos × dois feature sets. A validação cruzada reajusta o **pipeline inteiro** a cada fold — imputador e escalonador incluídos —, o que impede vazamento dentro da própria CV.

In [ ]:
tabela, resultados = executar_experimentos(dados, config)

colunas = ["modelo", "feature_set", "n_features", "cv_f2", "cv_recall",
           "cv_precision", "cv_f1", "cv_roc_auc", "tempo_s"]
tabela[colunas].sort_values("cv_f2", ascending=False).reset_index(drop=True)

In [ ]:
campeao = selecionar_campeao(tabela)
print(f"Critério: {CRITERIO_SELECAO}\n")
print(f"Campeão: {campeao['modelo']} ({campeao['feature_set']}, {campeao['n_features']} features)")
print(f"  F2:     {campeao['cv_f2']:.4f} ± {campeao['cv_f2_std']:.4f}")
print(f"  recall: {campeao['cv_recall']:.4f} ± {campeao['cv_recall_std']:.4f}")
print(f"  F1:     {campeao['cv_f1']:.4f}")
print(f"\nHiperparâmetros: {campeao['parametros']}")

### Leitura

**A regressão logística venceu** — e é também o modelo mais simples do catálogo. Três observações:

1. **A EDA previu isso.** A Fase 2 mostrou separação clara no PCA em duas dimensões, e a Fase 3 mostrou que Pearson e Spearman divergem no máximo 0,22, indicando relações predominantemente lineares. Um modelo linear tinha tudo para funcionar aqui.

2. **O baseline tem recall e F2 exatamente zero**, com accuracy de 62,7%. É a demonstração mais direta de que accuracy não pode orientar seleção neste problema: um modelo inútil obtém quase dois terços dela.

3. **As diferenças no topo são pequenas.** Entre a logística (F2 = 0,9685) e o XGBoost (0,9667) há 0,2 ponto percentual, enquanto o desvio na CV é de 1,2 pp. **Isso é empate estatístico**, não vitória. Com 341 amostras de treino divididas em 5 folds, cada fold valida ~68 casos, dos quais ~25 são malignos — um único caso vale 4 pontos percentuais de recall.

O terceiro ponto importa para a honestidade do relatório: declarar a logística "melhor que" o XGBoost seria superinterpretar. O correto é dizer que **ela é equivalente dentro do ruído e preferível por ser mais simples e mais interpretável** — o que é um argumento legítimo, só não é um argumento sobre desempenho.

In [ ]:
# Quão grande é o ruído, comparado às diferenças entre modelos?
topo = tabela[tabela["modelo"] != "dummy"].nlargest(6, "cv_f2")
analise = topo[["modelo", "feature_set", "cv_f2", "cv_f2_std"]].copy()
analise["diferenca_para_o_topo"] = (analise["cv_f2"].max() - analise["cv_f2"]).round(4)
analise["dentro_do_desvio"] = analise["diferenca_para_o_topo"] <= analise["cv_f2_std"].iloc[0]
analise.round(4).reset_index(drop=True)

---
## 4. Feature set completo versus reduzido

A Fase 3 construiu dois conjuntos: as 30 features originais e 24 após remover a redundância geométrica (`radius`/`perimeter`/`area`). A comparação foi deixada para a evidência empírica (ADR-009).

In [ ]:
pivo = tabela[tabela["modelo"] != "dummy"].pivot_table(
    index="modelo", columns="feature_set", values=["cv_f2", "cv_recall"]
)
pivo["delta_f2"] = (pivo[("cv_f2", "reduzido")] - pivo[("cv_f2", "completo")]).round(4)
pivo.round(4).sort_values("delta_f2")

### Leitura

O conjunto reduzido perde desempenho em **todos** os modelos, mas por margens pequenas — entre 0,1 e 2,1 pontos percentuais de F2. O maior prejuízo é na regressão logística (−2,1 pp), o menor no XGBoost (−0,8 pp) e na floresta (−0,1 pp).

O padrão faz sentido: **os modelos de árvore são os menos afetados**, porque já lidavam com a redundância internamente escolhendo uma feature do grupo e ignorando as demais. A regressão logística sofre mais porque, com regularização L2, o peso de uma informação redundante se distribui entre as features correlacionadas — remover parte delas concentra o peso de forma menos estável.

**O que isso decide.** As perdas estão dentro ou próximas do desvio da CV, então a diferença não é claramente real. Mas como também não há *ganho* de desempenho no conjunto reduzido, a escolha recai sobre o completo — ele é o que o critério selecionou, e a redução seria um custo sem contrapartida mensurável.

**Isso não anula o trabalho da Fase 3.** A análise de multicolinearidade permanece essencial para a Fase 6: o VIF de 3.904 significa que os coeficientes da logística são instáveis e que o SHAP vai diluir crédito entre `radius`, `perimeter` e `area`. Saber disso de antemão é o que permitirá interpretar as explicações corretamente, em vez de atribuir importância baixa a features que, em conjunto, são decisivas.

---
## 5. Calibração

A UI vai exibir "87% de probabilidade de malignidade". Essa afirmação só é honesta se, entre todos os casos a que o modelo atribui 87%, aproximadamente 87% forem de fato malignos.

Um modelo pode ordenar os casos perfeitamente (ROC-AUC alto) e ainda assim errar o *quanto* — acerta quem é mais suspeito, erra a magnitude. Para um sistema cuja zona cinzenta é definida por **faixas de probabilidade**, isso não é detalhe estético: probabilidades deslocadas encaminham os casos errados.

In [ ]:
colunas_campeao = construir_feature_sets(dados, config)[campeao["feature_set"]]
modelo = resultados[(campeao["modelo"], campeao["feature_set"])].estimador

proba_validacao = modelo.predict_proba(dados.x_validacao[colunas_campeao])[:, 1]
diagnostico = calibrate.diagnosticar(dados.y_validacao, proba_validacao)
print(diagnostico)

In [ ]:
diagnostico.para_tabela()

In [ ]:
comparacao_calibracao, melhor_calibrado = calibrate.comparar_metodos(
    modelo,
    dados.x_treino[colunas_campeao], dados.y_treino,
    dados.x_validacao[colunas_campeao], dados.y_validacao,
    config,
)
comparacao_calibracao.round(4)

### Leitura

**O modelo já está adequadamente calibrado**: erro médio de 3,0 pp, abaixo da tolerância de 5 pp adotada. A tabela faixa a faixa mostra concordância quase perfeita nos extremos.

A comparação entre métodos revela uma **tensão real entre critérios**:

- a regressão **isotônica melhora a confiabilidade** (erro médio e máximo) mas **piora o Brier score**;
- isso acontece porque ela ganha calibração às custas de **resolução** — as probabilidades ficam menos discriminantes.

Como o Brier é uma regra de pontuação própria e o erro do original já está dentro da tolerância, **mantemos as probabilidades originais** (ADR-015). Simplicidade é benefício adicional: uma camada a menos entre o modelo e a predição servida.

**Uma ressalva que importa.** O único desvio relevante está na faixa de ~76% prevista contra ~58% observada: o modelo é levemente otimista **justamente na região ambígua**. Com apenas 12 amostras naquela faixa, a estimativa é ruidosa — um caso vale 8 pp —, mas a direção é consistente com o esperado de um modelo em região de fronteira.

Isso **reforça** a necessidade da zona cinzenta em vez de contradizê-la: é exatamente ali que o sistema deve encaminhar para revisão em vez de rotular.

---
## 6. Limiar de decisão

O corte padrão de 0,5 presume que errar para mais e errar para menos custam o mesmo. Em oncologia, não custam.

**O limiar é escolhido na validação, nunca no teste.**

In [ ]:
limiar = escolher_limiar(
    dados.y_validacao, proba_validacao, config.modelagem.recall_minimo_alvo
)
print(limiar)

In [ ]:
# A troca, explícita.
linhas = []
for nome, corte in [("padrão (0,5)", 0.5), (f"ajustado ({limiar.limiar:.3f})", limiar.limiar)]:
    m = metricas_classificacao(dados.y_validacao, (proba_validacao >= corte).astype(int), proba_validacao)
    linhas.append({
        "limiar": nome,
        "recall": m["recall"],
        "precision": m["precision"],
        "especificidade": m["especificidade"],
        "cânceres não sinalizados (FN)": m["falsos_negativos"],
        "revisões desnecessárias (FP)": m["falsos_positivos"],
    })
pd.DataFrame(linhas).set_index("limiar").round(3)

### A troca, em termos clínicos

Baixar o limiar de 0,50 para 0,18 converte **1 câncer não sinalizado em 10 revisões desnecessárias**.

Essa frase é a decisão de projeto inteira. Vale a pena? No contexto de triagem oncológica, sim — e por uma razão que não é apenas de magnitude de custo:

- o **falso negativo sai do sistema**: o caso é despriorizado e ninguém olha de novo;
- o **falso positivo continua no sistema**: um profissional revisa, discorda, e o erro é corrigido no mesmo dia.

Os dois erros não apenas custam diferente — eles têm **mecanismos de correção diferentes**. Um é recuperável por desenho do fluxo de trabalho; o outro não.

A contrapartida precisa estar dita com honestidade: a precisão cai de 0,955 para 0,796, o que significa que **cerca de 1 em cada 5 casos sinalizados como suspeitos será benigno**. O sistema aumenta a carga de revisão. Ele só se justifica se a equipe tiver capacidade para absorver esse volume — e dimensionar isso é parte da decisão de adoção, não um detalhe de implementação.

In [ ]:
# Como a fila de triagem ficaria, com as três faixas.
faixas = classificar_com_faixa(proba_validacao, limiar.limiar, config)
resumo = resumo_faixas(faixas)

distribuicao = pd.DataFrame({
    "casos": pd.Series(resumo),
    "proporcao": (pd.Series(resumo) / len(faixas)).round(3),
})

# Quantos malignos reais caem em cada faixa?
checagem = pd.crosstab(faixas, dados.y_validacao.to_numpy(), rownames=["faixa"], colnames=["maligno"])
print(distribuicao.to_string())
print()
print(checagem.to_string())

### Leitura das faixas

A faixa cinzenta captura os casos que o modelo não consegue resolver — e a verificação cruzada mostra se ela está fazendo o trabalho certo: **nenhum caso maligno deve cair na faixa "benigno"**.

Note que a zona cinzenta tem precedência sobre o limiar na implementação: um caso com probabilidade 0,50 está acima do limiar de 0,18, mas dentro da zona 0,30–0,70, e é **encaminhado para revisão em vez de rotulado como maligno**. É conservador na direção certa — o sistema admite não saber, em vez de simular certeza.

---
## 7. Síntese

### Decisões da fase

| Decisão | Justificativa |
|---|---|
| **Seleção por F2**, não recall puro | Recall puro escolhe o degenerado; a regra de um erro padrão falha porque o degenerado tem desvio zero (ADR-014) |
| **Campeão: regressão logística, 30 features** | Melhor F2 e o mais simples; a EDA previu que linear bastaria |
| **Sem recalibração** | Erro médio de 3,0 pp já abaixo da tolerância; isotônica melhora confiabilidade mas piora Brier (ADR-015) |
| **Limiar 0,18**, não 0,5 | Atinge recall de 100% na validação; converte 1 FN em 10 FP |
| **Feature set completo** | Reduzido perde em todos os modelos, sem ganho compensatório |

### Números a reter

- Baseline: accuracy 62,7%, recall **0**, F2 **0**
- Campeão na CV: F2 = 0,9685 ± 0,0122, recall = 0,9686
- Diferença para o segundo colocado: **0,2 pp**, contra desvio de 1,2 pp — **empate estatístico**
- Limiar 0,18 na validação: recall 1,000 (0 FN), precisão 0,796 (11 FP)

### O que ainda não sabemos

- **Como o modelo se sai no teste.** Tudo acima foi medido em treino e validação. O teste permanece intocado até a Fase 6.
- Se o desempenho se mantém sob outras sementes — a análise de robustez é da Fase 6.
- Quais features o modelo realmente usa, e se bate com a expectativa da EDA (`concave points_worst` no topo, cinco features irrelevantes).
- Se a multicolinearidade vai distorcer a interpretação como previsto.